# Notebook 67 — Policy Learning

**Policy learning specifies adaptive controller candidates.**

Notebook 67 continues the learning-controller arc for the confidence-scheduled-decoding repository. Notebook 61 converted runtime telemetry into validated state-action-outcome records. Notebook 67 uses those records to learn bounded policy candidates, rank them offline, and defer deployment to later safety checks.

In [ ]:

from pathlib import Path
import json, zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import ListedColormap, BoundaryNorm

OUT = Path("figures")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (14, 8),
    "font.size": 18,
    "axes.titlesize": 34,
    "axes.labelsize": 24,
    "legend.fontsize": 15,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
})

rng = np.random.default_rng(67)

def savefig(name):
    path = OUT / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    return path

def pill(ax, xy, w, h, text, fs=18, lw=2.2):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.12",
        linewidth=lw, edgecolor="black", facecolor="white"
    )
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fs)
    return box

def arrow(ax, start, end, lw=2.0, mutation=18):
    arr = FancyArrowPatch(
        start, end, arrowstyle="->", mutation_scale=mutation,
        linewidth=lw, color="black", shrinkA=4, shrinkB=4
    )
    ax.add_patch(arr)
    return arr

# Synthetic confidence-scheduled decoding telemetry.
n = 360
t = np.arange(n)
arrival = 0.45 + 0.25*np.sin(t/38) + 0.10*rng.normal(size=n)
arrival = np.clip(arrival, 0.05, 0.95)

confidence = 0.52 + 0.28*np.sin(t/53 + 0.9) + 0.13*rng.normal(size=n)
confidence = np.clip(confidence, 0.05, 0.98)

queue = 0.35 + 0.40*arrival - 0.18*confidence + 0.10*rng.normal(size=n)
queue = np.clip(queue, 0.04, 0.95)

util = 0.52 + 0.37*arrival + 0.08*rng.normal(size=n)
util = np.clip(util, 0.06, 0.98)

memory = 0.45 + 0.30*queue + 0.10*rng.normal(size=n)
memory = np.clip(memory, 0.05, 0.98)

reserve = 1.0 - 0.62*util - 0.18*memory + 0.08*rng.normal(size=n)
reserve = np.clip(reserve, 0.04, 0.95)

verification = 0.35 + 0.45*(1-confidence) + 0.08*rng.normal(size=n)
verification = np.clip(verification, 0.03, 0.95)

S = np.vstack([queue, util, memory, arrival, reserve, verification, confidence]).T

actions = np.array(["steady", "shorten", "verify", "reserve", "scale-out"])
def expert_policy(s):
    q,u,m,a,r,v,c = s
    if c < 0.35 and q > 0.55:
        return "verify"
    if u > 0.82 or q > 0.78:
        return "shorten"
    if r < 0.22 and u > 0.68:
        return "scale-out"
    if r > 0.65 and q < 0.45:
        return "reserve"
    return "steady"

labels = np.array([expert_policy(row) for row in S])
action_to_idx = {a:i for i,a in enumerate(actions)}
y_action = np.array([action_to_idx[a] for a in labels])

# Offline outcome components for policy candidates.
candidates = np.array(["baseline", "conservative", "balanced", "aggressive", "unsafe"])
gain = np.array([0.52, 0.57, 0.72, 0.83, 0.92])
latency_cost = -np.array([0.25, 0.21, 0.24, 0.38, 0.62])
spillover_cost = -np.array([0.18, 0.14, 0.16, 0.25, 0.48])
violation_cost = -np.array([0.06, 0.03, 0.05, 0.16, 0.38])
net_score = gain + latency_cost + spillover_cost + violation_cost

# Learned policy proxy surface.
x = np.linspace(0.05, 0.95, 120)
y = np.linspace(0.05, 0.95, 120)
R, U = np.meshgrid(x, y)
steady_score = 0.70 - 1.8*U - 0.35*np.abs(R-0.45)
reserve_score = 0.15 + 1.45*R - 1.15*U - 0.35*(R-0.75)**2
rebalance_score = 0.38 + 1.15*U + 0.45*R - 0.55*np.abs(U-0.60)
scale_score = -0.1 + 1.35*U + 0.55*R - 0.55*np.abs(R-0.65)
shorten_score = -0.2 + 2.2*U - 1.15*R
shorten_score += 0.60*(U > 0.78)*(R < 0.22)
scores = np.stack([steady_score, reserve_score, rebalance_score, scale_score, shorten_score])
Z = np.argmax(scores, axis=0)
surface_labels = ["steady", "reserve", "rebalance", "scale-out", "shed/shorten"]
cmap = ListedColormap(["#440154", "#3b528b", "#21918c", "#5ec962", "#fde725"])
norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), cmap.N)


## 1. Notebook position

Notebook 67 sits after telemetry dataset construction and before offline evaluation. The key boundary is important: learning produces **candidate policies**, not immediate runtime authority.

In [ ]:

fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Notebook 67 continues the learning-controller arc", pad=25)

top = [("43\nResource\nAllocation", .06), ("49\nAdaptive\nInfrastructure", .29), ("53\nDistributed\nScheduling", .52), ("59\nCluster\nOptimization", .75)]
bot = [("61\nTelemetry\nDataset", .03), ("67\nPolicy\nLearning", .25), ("71\nOffline\nEvaluation", .47), ("73\nSafety\nBounds", .69), ("79\nAdaptive\nController", .89)]

for txt, x0 in top:
    pill(ax, (x0, .67), .20, .13, txt, fs=16)
for i in range(len(top)-1):
    arrow(ax, (top[i][1]+.20, .735), (top[i+1][1], .735))
ax.text(.5, .61, "second arc: controller → infrastructure → distributed scheduling → cluster optimization", ha="center", fontsize=19)
ax.plot([.08, .92], [.56, .56], color="black", lw=2)

for txt, x0 in bot:
    pill(ax, (x0, .30), .19, .14, txt, fs=16)
for i in range(len(bot)-1):
    arrow(ax, (bot[i][1]+.19, .37), (bot[i+1][1], .37))
arrow(ax, (.80, .67), (.35, .44), lw=2.2, mutation=20)
ax.text(.5, .52, "optimized serving traces become training data", ha="center", fontsize=18)
ax.text(.5, .23, "third arc: telemetry dataset → policy learning → evaluation → safety → controller", ha="center", fontsize=20)

savefig("67_third_arc_position_v3.png")


## 2. Learning pipeline

The validated dataset is converted into state-action records, policy candidates, and offline scores. This keeps the workflow auditable: the learner proposes; later notebooks evaluate and bound.

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Learning dataset specifies policy optimization", pad=24)

steps = [
    ("validated\ndataset D", .02),
    ("state-action\nrecords", .19),
    ("policy\nlearner", .36),
    ("candidate\npolicy πθ", .53),
    ("offline\nscore", .70),
    ("controller\ncandidate", .86),
]
for txt, x0 in steps:
    pill(ax, (x0, .50), .16, .15, txt, fs=18)
for i in range(len(steps)-1):
    arrow(ax, (steps[i][1]+.16, .575), (steps[i+1][1], .575), lw=2.2)

pill(ax, (.20, .14), .60, .12, "Learning produces candidates for offline evaluation, not immediate deployment.", fs=19)
savefig("67_learning_pipeline_v3.png")


## 3. State vector to policy candidates

The controller input is a compact state vector built from queue depth, utilization, memory pressure, arrival rate, reserve capacity, verification load, and confidence.

In [ ]:

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("State vector maps telemetry into policy candidates", pad=20)

pill(ax, (.37, .54), .26, .16, "State vector\nS(t)", fs=24)
pill(ax, (.37, .34), .26, .13, "policy model\nπθ(S)", fs=21)
pill(ax, (.37, .16), .26, .12, "action scores\nπθ(a | S)", fs=19)
pill(ax, (.40, .01), .20, .10, "candidate action", fs=18)

items = [
    ("q(t)\nqueue", (.07, .62), (.37, .61)),
    ("u(t)\nutilization", (.40, .82), (.50, .70)),
    ("m(t)\nmemory", (.72, .62), (.63, .61)),
    ("λ(t)\narrival", (.07, .36), (.37, .57)),
    ("r(t)\nreserve", (.40, .20), (.50, .54)),
    ("v(t)\nverification", (.72, .36), (.63, .57)),
]
for txt, xy, endpoint in items:
    pill(ax, xy, .21, .11, txt, fs=17)
    arrow(ax, (xy[0]+.105, xy[1]+.055), endpoint, lw=1.8, mutation=15)
arrow(ax, (.50, .54), (.50, .47), lw=2.3)
arrow(ax, (.50, .34), (.50, .28), lw=2.3)
arrow(ax, (.50, .16), (.50, .11), lw=2.3)
ax.text(.5, -.04, r"$S(t)=[q(t), u(t), m(t), \lambda(t), r(t), v(t), c(t)]$", ha="center", fontsize=25)
savefig("67_policy_model_inputs_v3.png")


## 4. Learned candidate surface

This synthetic surface shows the policy learner mapping reserve capacity and active load into candidate actions. The important point is not the exact boundary; it is that the candidate surface is bounded and interpretable before runtime use.

In [ ]:

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(Z, origin="lower", extent=[0.05, 0.95, 0.05, 0.95], cmap=cmap, norm=norm, aspect="auto")
ax.set_title("Learned policy candidate surface", pad=20)
ax.set_xlabel("reserve capacity")
ax.set_ylabel("active load")
for label, px, py in [
    ("steady", .18, .22),
    ("reserve", .73, .24),
    ("rebalance", .55, .56),
    ("scale-out", .78, .82),
    ("shed /\nshorten", .17, .75),
]:
    ax.text(px, py, label, ha="center", va="center", fontsize=22)
cbar = fig.colorbar(im, ax=ax, ticks=np.arange(5))
cbar.ax.set_yticklabels(surface_labels)
cbar.set_label("selected candidate action")
savefig("67_policy_candidate_surface_v3.png")


## 5. Learning objective

The learner balances imitation fit, estimated value, constraint penalties, and imbalance penalties. This explicitly prevents the training objective from rewarding unsafe shifts merely because they improve throughput.

In [ ]:

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Learning objective balances fit, value, and constraints", pad=24)

terms = [
    ("prediction\nloss", .10),
    ("value\nloss", .28),
    ("constraint\npenalty", .46),
    ("imbalance\npenalty", .64),
    ("regularized\nobjective", .82),
]
for txt, x0 in terms:
    pill(ax, (x0, .55), .15, .12, txt, fs=17)
for i in range(len(terms)-1):
    arrow(ax, (terms[i][1]+.15, .61), (terms[i+1][1], .61), lw=2.0)

ax.text(.5, .34, r"$\mathcal{L}(\theta)=\ell(\pi_\theta,\pi_0)-\alpha \widehat{U}(\pi_\theta)+\beta C(\pi_\theta)+\gamma R(\pi_\theta)$",
        ha="center", fontsize=26)
pill(ax, (.22, .12), .56, .10, "The learner is rewarded for useful policy shifts and penalized for unsafe or unstable ones.", fs=17)
savefig("67_loss_objective_v3.png")


## 6. Constraint regularization

An unregularized objective pushes toward aggressive adaptation. The regularized objective selects a bounded adaptation intensity where throughput improvement does not overwhelm latency and constraint risk.

In [ ]:

x = np.linspace(0, 1, 300)
throughput_gain = 1.12*(1 - np.exp(-3.0*x))
latency_pressure = 0.18 + 0.75*(x**3.2)
constraint_risk = 0.10 + 0.12*np.maximum(0, x-.65)**2*8.0
unregularized = throughput_gain - latency_pressure - constraint_risk
regularized = unregularized - 1.25*(x**3.4) - 0.22*np.maximum(0, x-.72)**2*8.0
selected = x[np.argmax(regularized)]

fig, ax = plt.subplots(figsize=(13, 8))
ax.plot(x, throughput_gain, label="throughput gain")
ax.plot(x, latency_pressure, label="latency pressure")
ax.plot(x, constraint_risk, label="constraint risk")
ax.plot(x, unregularized, label="unregularized objective")
ax.plot(x, regularized, linewidth=3, label="regularized objective")
ax.axvline(selected, color="black", linestyle="--", label=f"selected ≈ {selected:.2f}")
ax.set_title("Constraint regularization avoids unsafe adaptation", pad=10)
ax.set_xlabel("policy adaptation intensity")
ax.set_ylabel("normalized value")
ax.grid(alpha=.25)
ax.legend(loc="upper left")
savefig("67_constraint_regularization_v3.png")


## 7. Counterfactual ranking

Candidate policies are scored offline with throughput gain, latency cost, spillover cost, and violation cost. The best raw gain can be rejected when its constraint costs dominate.

In [ ]:

fig, ax = plt.subplots(figsize=(13, 8))
idx = np.arange(len(candidates))
w = 0.18
ax.bar(idx - 1.5*w, gain, width=w, label="gain")
ax.bar(idx - 0.5*w, latency_cost, width=w, label="latency cost")
ax.bar(idx + 0.5*w, spillover_cost, width=w, label="spillover cost")
ax.bar(idx + 1.5*w, net_score, width=w, label="net score")
ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(idx)
ax.set_xticklabels(candidates, rotation=20)
ax.set_title("Counterfactual ranking selects bounded policy candidates", pad=12)
ax.set_ylabel("offline normalized value")
ax.grid(axis="y", alpha=.25)
ax.legend(loc="upper left")
savefig("67_counterfactual_ranking_v3.png")


## 8. Validation bounds

Offline training requires generalization checks. The selected epoch should be justified by validation and test behavior, not just training loss.

In [ ]:

epochs = np.arange(1, 41)
train_loss = 0.78*np.exp(-epochs/15) + 0.10
val_loss = 0.74*np.exp(-epochs/17) + 0.20
test_loss = 0.72*np.exp(-epochs/18) + 0.26
selected_epoch = 40

fig, ax = plt.subplots(figsize=(13, 8))
ax.plot(epochs, train_loss, label="train loss")
ax.plot(epochs, val_loss, label="validation loss")
ax.plot(epochs, test_loss, label="test loss")
ax.axvline(selected_epoch, color="black", linestyle="--", label=f"selected epoch {selected_epoch}")
ax.set_title("Validation performance bounds policy learning", pad=12)
ax.set_xlabel("training epoch")
ax.set_ylabel("loss")
ax.grid(alpha=.25)
ax.legend(loc="lower left")
savefig("67_generalization_gap_v3.png")


## 9. Policy logits from telemetry states

The learned policy should expose action preferences over real telemetry windows. This heatmap gives a notebook-style diagnostic: whether the policy places probability mass on sensible actions across the dataset.

In [ ]:

sample_idx = np.linspace(0, n-1, 80).astype(int)
sub = S[sample_idx]
Q,U,M,A,R,V,C = sub.T
logits = np.vstack([
    1.2 - 1.5*U - .4*Q,                      # steady
    .2 + 1.5*R - .9*U,                        # reserve
    .3 + 1.8*(1-C) + .7*V,                    # verify
    .2 + 1.6*U + .8*Q - .9*R,                 # shorten
    -.1 + 1.4*U + .9*(.35-R),                 # scale-out
])
logits = logits - logits.max(axis=0, keepdims=True)
probs = np.exp(logits) / np.exp(logits).sum(axis=0, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(probs, aspect="auto", origin="lower", vmin=0, vmax=1)
ax.set_title("Policy probabilities over telemetry windows", pad=14)
ax.set_xlabel("sampled telemetry window")
ax.set_ylabel("candidate action")
ax.set_yticks(np.arange(len(actions)))
ax.set_yticklabels(["steady", "reserve", "verify", "shorten", "scale-out"])
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("πθ(a | S)")
savefig("67_policy_probability_heatmap_v3.png")


## 10. Final synthesis

Notebook 67 takes validated telemetry and turns it into bounded policy candidates. Notebook 71 should then evaluate those candidates offline before Notebook 73 converts them into safety-bounded controllers.

In [ ]:

fig, ax = plt.subplots(figsize=(15, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Policy learning specifies adaptive controller candidates", pad=25)

steps = [
    ("validated\ntelemetry", .04),
    ("state-action\nrecords", .20),
    ("outcome\nlabels", .36),
    ("regularized\npolicy learner", .52),
    ("counterfactual\nranking", .70),
    ("bounded\ncandidate", .87),
]
for txt, x0 in steps:
    pill(ax, (x0, .52), .14, .14, txt, fs=16)
for i in range(len(steps)-1):
    arrow(ax, (steps[i][1]+.14, .59), (steps[i+1][1], .59), lw=2.2)

pill(ax, (.19, .18), .62, .14,
     "Notebook 67 learns candidate policies from telemetry records before runtime control.",
     fs=18)
savefig("67_final_synthesis_v3.png")


## Download figures

Run the cell below after generating figures. It creates a zip file containing all Notebook 67 v3 PNG outputs.

In [ ]:

zip_path = Path("notebook_67_figures_v3.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("67_*_v3.png")):
        zf.write(p, arcname=p.name)

manifest = {
    "notebook": "67_policy_learning_v3",
    "figures": [p.name for p in sorted(OUT.glob("67_*_v3.png"))],
    "zip": str(zip_path),
}
Path("notebook_67_manifest_v3.json").write_text(json.dumps(manifest, indent=2))
manifest
